# 04 — Nelder-Mead

Nelder-Mead je metoda optimizacije bez izvoda koja radi nad kontinualnim vrednostima. Kako naši hiperparametri nisu uvek kontinualni (npr. `n_estimators` mora biti ceo broj), moramo sami da napravimo funkciju koja "prevodi" kontinualni vektor `x` u prave hiperparametre modela — to radimo u funkcijama `dekoduj_svm` i `dekoduj_rf` ispod.

Ne ispisujemo svaku pojedinačnu evaluaciju — u svesci prikazujemo samo rezultat po seed-u i finalni prosek.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import time

In [2]:
breast_cancer = load_breast_cancer()
digits = load_digits()

X_bc, y_bc = breast_cancer.data, breast_cancer.target
X_dg, y_dg = digits.data, digits.target

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc)
X_dg_train, X_dg_test, y_dg_train, y_dg_test = train_test_split(
    X_dg, y_dg, test_size=0.2, random_state=42, stratify=y_dg)

skaler_bc = StandardScaler()
X_bc_train = skaler_bc.fit_transform(X_bc_train)
X_bc_test = skaler_bc.transform(X_bc_test)

skupovi_podataka = {
    'breast_cancer': (X_bc_train, y_bc_train),
    'digits': (X_dg_train, y_dg_train),
}

SEEDOVI = list(range(10))
BROJ_FOLDOVA = 3  

## Funkcije za dekodiranje hiperparametara

In [3]:
def dekoduj_svm(x):
    C = 10 ** x[0]
    gamma = 10 ** x[1]
    C = np.clip(C, 1e-4, 1e4)
    gamma = np.clip(gamma, 1e-6, 1e2)
    return {'C': float(C), 'gamma': float(gamma)}


def dekoduj_rf(x):
    n_estimators = int(np.clip(round(x[0]), 10, 500))
    max_depth = int(np.clip(round(x[1]), 2, 50))
    min_samples_split = int(np.clip(round(x[2]), 2, 20))
    return {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
    }

## Funkcija cilja

`scipy.optimize.minimize` uvek *minimizuje* funkciju, a mi želimo da *maksimizujemo* tačnost (accuracy) — zato funkcija cilja vraća `-score`. 

In [4]:
def napravi_funkciju_cilja(ime_modela, X, y, kf, seed):
    def funkcija_cilja(x):
        if ime_modela == 'svm':
            params = dekoduj_svm(x)
            model = SVC(kernel='rbf', random_state=42, **params)
        else:
            params = dekoduj_rf(x)
            model = RandomForestClassifier(random_state=seed, n_jobs=-1, **params)

        score = cross_val_score(model, X, y, cv=kf, scoring='accuracy', n_jobs=-1).mean()
        return -score  #minimize trazi minimum, a mi hocemo maksimum tacnosti

    return funkcija_cilja

## Pokretanje 

Početne tačke (`x0`) su iste za svaki seed (menja se samo podela na foldove i, kod Random Forest-a, `random_state`).

In [5]:
pocetne_tacke = {
    'svm': np.array([0.0, -2.0]),          # C=10^0=1, gamma=10^-2=0.01
    'random_forest': np.array([100.0, 10.0, 2.0]),
}

simplex_rf = np.array([
    pocetne_tacke['random_forest'],
    pocetne_tacke['random_forest'] + np.array([20.0, 0.0, 0.0]),  
    pocetne_tacke['random_forest'] + np.array([0.0, 5.0, 0.0]),   
    pocetne_tacke['random_forest'] + np.array([0.0, 0.0, 3.0]),  
])

rezultati_nm = []

for ime_skupa, (X, y) in skupovi_podataka.items():
    for ime_modela in ['svm', 'random_forest']:

        skorovi_po_seedu = []
        vremena_po_seedu = []
        parametri_po_seedu = []
        evaluacije_po_seedu = []

        x0 = pocetne_tacke[ime_modela]

        for seed in SEEDOVI:
            kf = KFold(n_splits=BROJ_FOLDOVA, shuffle=True, random_state=seed)
            funkcija_cilja = napravi_funkciju_cilja(ime_modela, X, y, kf, seed)

            opcije = {'maxiter': 60, 'xatol': 1e-2, 'fatol': 1e-5}
            if ime_modela == 'random_forest':
                opcije['initial_simplex'] = simplex_rf

            pocetak = time.time()
            rezultat = minimize(funkcija_cilja, x0, method='Nelder-Mead', options=opcije)
            trajanje = time.time() - pocetak

            if ime_modela == 'svm':
                najbolji_params = dekoduj_svm(rezultat.x)
            else:
                najbolji_params = dekoduj_rf(rezultat.x)

            skorovi_po_seedu.append(-rezultat.fun)
            vremena_po_seedu.append(trajanje)
            parametri_po_seedu.append(najbolji_params)
            evaluacije_po_seedu.append(rezultat.nfev)

        indeks_najboljeg = int(np.argmax(skorovi_po_seedu))

        rezultati_nm.append({
            'dataset': ime_skupa,
            'model': ime_modela,
            'method': 'nelder_mead',
            'mean_score': np.mean(skorovi_po_seedu),
            'std_score': np.std(skorovi_po_seedu),
            'n_evaluations': int(np.mean(evaluacije_po_seedu)),
            'mean_time_sec': np.mean(vremena_po_seedu),
            'best_params': parametri_po_seedu[indeks_najboljeg],
        })

        print(f'{ime_skupa:15s} {ime_modela:15s} '
              f'mean_score={np.mean(skorovi_po_seedu):.4f} (+/- {np.std(skorovi_po_seedu):.4f})  '
              f'mean_evals={np.mean(evaluacije_po_seedu):5.1f}  '
              f'mean_time={np.mean(vremena_po_seedu):6.1f}s')

breast_cancer   svm             mean_score=0.9738 (+/- 0.0028)  mean_evals= 18.2  mean_time=   0.5s
breast_cancer   random_forest   mean_score=0.9622 (+/- 0.0019)  mean_evals= 65.9  mean_time=  18.8s
digits          svm             mean_score=0.9909 (+/- 0.0018)  mean_evals= 32.1  mean_time=   2.7s
digits          random_forest   mean_score=0.9747 (+/- 0.0016)  mean_evals= 82.8  mean_time=  26.9s


## Rezultati

In [6]:
df_nm = pd.DataFrame(rezultati_nm)
df_nm

,dataset,model,method,mean_score,std_score,n_evaluations,mean_time_sec,best_params
0,breast_cancer,svm,nelder_mead,0.973844,0.002844,18,0.500959,"{'C': 1.0005758119893609, 'gamma': 0.012589254..."
1,breast_cancer,random_forest,nelder_mead,0.962189,0.001916,65,18.805515,"{'n_estimators': 100, 'max_depth': 10, 'min_sa..."
2,digits,svm,nelder_mead,0.990884,0.001827,32,2.655809,"{'C': 0.9968389521625052, 'gamma': 0.000794328..."
3,digits,random_forest,nelder_mead,0.974669,0.001562,82,26.930925,"{'n_estimators': 118, 'max_depth': 12, 'min_sa..."


In [7]:
kolone_rezultata = ['dataset', 'model', 'method', 'mean_score', 'std_score',
                    'n_evaluations', 'mean_time_sec', 'best_params']

import os
os.makedirs('./results', exist_ok=True)

df_nm[kolone_rezultata].to_csv('./results/all_results.csv', mode='a', header=False, index=False)
print('Rezultati sačuvani u results/all_results.csv')

Rezultati sačuvani u results/all_results.csv
